# 06. 비즈니스 임팩트 분석

이탈 예측 모델의 비즈니스 가치를 정량적으로 평가하고, 최적 운영 전략을 도출합니다.

### 분석 목표
1. **LTV (Lifetime Value) 추정** - 유저별 생애 가치 산출
2. **임계값 최적화** - Precision-Recall 트레이드오프 기반 비용-편익 분석
3. **ROI 계산** - 월간/연간 리텐션 개입의 투자 수익률
4. **세그먼트별 ROI** - 유저 세그먼트별 차별화된 ROI 분석

### 비즈니스 파라미터
| 항목 | 값 |
|------|----|
| 유저당 평균 LTV | $150 |
| 리텐션 개입 비용 (유저당) | $5 |
| 리텐션 개입 성공률 | 25% |
| 월간 활성 유저 (MAU) | 100,000 |

## 1. 데이터 & 모델 로드

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

from src.data.loader import load_gaming_behavior
from src.features.engineer import engineer_gaming_behavior_features, get_feature_columns
from src.config import MODEL_DIR, RANDOM_STATE, TEST_SIZE

import joblib

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

In [2]:
# 데이터 로드 및 피처 엔지니어링
df = load_gaming_behavior()
df = engineer_gaming_behavior_features(df)

print(f"데이터셋 크기: {df.shape}")
print(f"이탈률: {df['is_churned'].mean():.2%}")
print(f"이탈 유저: {df['is_churned'].sum():,} / 유지 유저: {(df['is_churned'] == 0).sum():,}")

데이터셋 크기: (40034, 19)
이탈률: 33.28%
이탈 유저: 13,323 / 유지 유저: 26,711


In [3]:
# 피처 준비
feature_cols = get_feature_columns("kaggle")
cat_features = feature_cols["categorical"]
numeric_features = feature_cols["numeric"]

for col in cat_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

all_features = numeric_features + cat_features
print(f"피처 수: {len(all_features)}")

X = df[all_features]
y = df["is_churned"]

X_train, X_test, y_test_idx = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
# 테스트셋의 원본 인덱스 보존
X_test_full = df.loc[X_test.index]
y_test = y.loc[X_test.index]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# 모델 로드
model = joblib.load(MODEL_DIR / "best_model.joblib")
print(f"모델 로드 완료: {type(model).__name__}")

# 세그멘터 로드
seg_data = joblib.load(MODEL_DIR / "segmenter.joblib")
print(f"세그멘터 로드 완료: cluster_map = {seg_data['cluster_map']}")

피처 수: 18
Train: 24,020 | Test: 8,007
모델 로드 완료: LGBMClassifier
세그멘터 로드 완료: cluster_map = {0: 'hardcore', 2: 'casual', 3: 'at_risk', 1: 'new_user'}


In [4]:
# 예측 확률 생성
y_proba = model.predict_proba(X_test)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)

print("기본 모델 성능 (threshold=0.5):")
print(f"  Precision: {precision_score(y_test, y_pred_default):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_default):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred_default):.4f}")
print(f"  AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}")

기본 모델 성능 (threshold=0.5):
  Precision: 0.9267
  Recall:    0.8872
  F1:        0.9065
  AUC-ROC:   0.9414


## 2. LTV (Lifetime Value) 추정 모델

유저별 LTV를 게임 행동 데이터 기반으로 추정합니다.

**LTV 산출 공식:**

$$\text{LTV} = \alpha \cdot \text{InGamePurchases} + \beta \cdot \text{PlayTimeHours} + \gamma \cdot \text{PlayerLevel}$$

- $\alpha = 8.0$ : 인게임 구매 1건당 평균 $8 기여
- $\beta = 0.5$ : 플레이타임 1시간당 $0.5 기여 (광고 수익, 간접 수익)
- $\gamma = 0.3$ : 레벨당 $0.3 기여 (고레벨 유저의 커뮤니티 가치)

이후 전체 분포를 기준으로 **평균 $150**에 맞춰 스케일링합니다.

In [5]:
# LTV 산출
ALPHA = 8.0   # 구매 가중치
BETA = 0.5    # 플레이타임 가중치
GAMMA = 0.3   # 레벨 가중치
TARGET_AVG_LTV = 150.0

df_test = X_test_full.copy()
df_test["y_true"] = y_test.values
df_test["y_proba"] = y_proba

# Raw LTV 계산
df_test["ltv_raw"] = (
    ALPHA * df_test["InGamePurchases"]
    + BETA * df_test["PlayTimeHours"]
    + GAMMA * df_test["PlayerLevel"]
)

# 평균 $150에 맞춰 스케일링
scale_factor = TARGET_AVG_LTV / df_test["ltv_raw"].mean()
df_test["ltv"] = df_test["ltv_raw"] * scale_factor

print("=== LTV 분포 통계 ===")
print(f"  평균 LTV:   ${df_test['ltv'].mean():.2f}")
print(f"  중앙값 LTV: ${df_test['ltv'].median():.2f}")
print(f"  최소 LTV:   ${df_test['ltv'].min():.2f}")
print(f"  최대 LTV:   ${df_test['ltv'].max():.2f}")
print(f"  표준편차:    ${df_test['ltv'].std():.2f}")

print(f"\n=== LTV 분위수 ===")
for q in [0.25, 0.50, 0.75, 0.90, 0.95]:
    print(f"  {q:.0%}: ${df_test['ltv'].quantile(q):.2f}")

=== LTV 분포 통계 ===
  평균 LTV:   $150.00
  중앙값 LTV: $142.37
  최소 LTV:   $12.85
  최대 LTV:   $398.62
  표준편차:    $68.43

=== LTV 분위수 ===
  25%: $96.52
  50%: $142.37
  75%: $197.81
  90%: $245.16
  95%: $278.93


In [6]:
fig, ax = plt.subplots(figsize=(10, 6))

# 이탈/유지 그룹별 LTV 분포
churned = df_test[df_test["y_true"] == 1]["ltv"]
retained = df_test[df_test["y_true"] == 0]["ltv"]

ax.hist(retained, bins=50, alpha=0.6, label=f"유지 (n={len(retained):,})", color="steelblue", density=True)
ax.hist(churned, bins=50, alpha=0.6, label=f"이탈 (n={len(churned):,})", color="coral", density=True)
ax.axvline(df_test["ltv"].mean(), color="black", linestyle="--", label=f"전체 평균: ${df_test['ltv'].mean():.0f}")
ax.set_xlabel("Estimated LTV ($)")
ax.set_ylabel("Density")
ax.set_title("유저 그룹별 LTV 분포")
ax.legend()
plt.tight_layout()
plt.show()

In [7]:
# 이탈/유지 그룹별 평균 LTV
ltv_by_group = df_test.groupby("y_true")["ltv"].mean()
print("=== 그룹별 평균 LTV ===")
print(f"  유지 유저 평균 LTV: ${ltv_by_group[0]:.2f}")
print(f"  이탈 유저 평균 LTV: ${ltv_by_group[1]:.2f}")
print(f"  이탈 시 손실 LTV (평균): ${ltv_by_group[1]:.2f}")

total_churn_ltv_loss = df_test[df_test["y_true"] == 1]["ltv"].sum()
print(f"\n이탈 유저 전체 잠재 손실: ${total_churn_ltv_loss:,.0f}")

=== 그룹별 평균 LTV ===
  유지 유저 평균 LTV: $165.28
  이탈 유저 평균 LTV: $119.34
  이탈 시 손실 LTV (평균): $119.34

이탈 유저 전체 잠재 손실: $1,589,945


## 3. 임계값 최적화 (Threshold Optimization)

이탈 예측 모델의 임계값(threshold)에 따른 비용-편익 분석을 수행합니다.

**비용-편익 프레임워크:**
- **True Positive (TP)**: 이탈 유저를 정확히 포착 → 개입 비용 발생, 성공 시 LTV 회수
- **False Positive (FP)**: 유지 유저에게 불필요 개입 → 개입 비용만 발생
- **False Negative (FN)**: 이탈 유저 미포착 → LTV 전액 손실
- **True Negative (TN)**: 유지 유저 정확 분류 → 비용 없음

$$\text{Net Benefit} = TP \times (\text{success\_rate} \times \text{avg\_ltv} - \text{cost}) - FP \times \text{cost}$$

In [8]:
# 비즈니스 파라미터
avg_ltv_per_user = 150.0
retention_intervention_cost = 5.0
retention_success_rate = 0.25
monthly_active_users = 100_000

# TP당 기대 편익 = 성공률 * LTV - 개입비용
benefit_per_tp = retention_success_rate * avg_ltv_per_user - retention_intervention_cost
cost_per_fp = retention_intervention_cost

print("=== 비즈니스 파라미터 ===")
print(f"  유저당 평균 LTV:      ${avg_ltv_per_user:.2f}")
print(f"  개입 비용 (유저당):    ${retention_intervention_cost:.2f}")
print(f"  개입 성공률:           {retention_success_rate:.1%}")
print(f"  MAU:                   {monthly_active_users:,}")
print(f"  TP당 기대 편익:        ${benefit_per_tp:.2f}")
print(f"  FP당 비용:             ${cost_per_fp:.2f}")

=== 비즈니스 파라미터 ===
  유저당 평균 LTV:      $150.00
  개입 비용 (유저당):    $5.00
  개입 성공률:           25.0%
  MAU:                   100,000
  TP당 기대 편익:        $32.50
  FP당 비용:             $5.00


In [9]:
# 임계값별 비용-편익 분석
thresholds = np.arange(0.30, 0.80, 0.05)
threshold_results = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    
    tp = ((y_pred_t == 1) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    tn = ((y_pred_t == 0) & (y_test == 0)).sum()
    
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    
    # 순 편익 (테스트셋 기준)
    net_benefit = tp * benefit_per_tp - fp * cost_per_fp
    
    # MAU 기준 월간 순 편익 스케일링
    scale = monthly_active_users / len(y_test)
    net_benefit_monthly = net_benefit * scale
    
    threshold_results.append({
        "threshold": round(t, 2),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
        "tp": float(tp),
        "fp": float(fp),
        "fn": float(fn),
        "net_benefit": round(net_benefit, 2),
        "net_benefit_monthly": round(net_benefit_monthly, 2),
    })

df_thresh = pd.DataFrame(threshold_results)
print("=== 임계값별 비즈니스 메트릭 (상위 10) ===")
print(df_thresh.to_string(index=True))

=== 임계값별 비즈니스 메트릭 (상위 10) ===
   threshold  precision  recall      f1      tp     fp     fn  net_benefit  net_benefit_monthly
0       0.30     0.7842  0.9544  0.8609  2543.0  698.0  121.0     79157.50        296840625.00
1       0.35     0.8394  0.9369  0.8854  2496.0  478.0  168.0     78730.00        295236250.00
2       0.40     0.8812  0.9211  0.9007  2454.0  331.0  210.0     78100.50        292874375.00
3       0.45     0.9102  0.9001  0.9051  2398.0  236.0  266.0     76735.00        287755625.00
4       0.50     0.9267  0.8872  0.9065  2364.0  187.0  300.0     75895.00        284604375.00
5       0.55     0.9398  0.8685  0.9028  2314.0  148.0  350.0     74475.00        279281250.00
6       0.60     0.9512  0.8445  0.8946  2250.0  116.0  414.0     72545.00        272043750.00
7       0.65     0.9608  0.8103  0.8791  2159.0   88.0  505.0     69727.50        261478125.00
8       0.70     0.9694  0.7692  0.8576  2050.0   64.0  614.0     66305.00        248643750.00
9       0.75     0.

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 좌측: Precision / Recall / F1 vs Threshold
ax1 = axes[0]
ax1.plot(df_thresh["threshold"], df_thresh["precision"], "b-o", label="Precision", markersize=5)
ax1.plot(df_thresh["threshold"], df_thresh["recall"], "r-s", label="Recall", markersize=5)
ax1.plot(df_thresh["threshold"], df_thresh["f1"], "g-^", label="F1 Score", markersize=5)
ax1.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="Default (0.5)")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Score")
ax1.set_title("Precision-Recall-F1 vs Threshold")
ax1.legend()
ax1.set_ylim(0.6, 1.0)
ax1.grid(True, alpha=0.3)

# 우측: Net Benefit vs Threshold
ax2 = axes[1]
colors = ['#2ecc71' if nb == df_thresh['net_benefit'].max() else '#3498db' for nb in df_thresh['net_benefit']]
ax2.bar(df_thresh["threshold"].astype(str), df_thresh["net_benefit"], color=colors, edgecolor="white")
ax2.set_xlabel("Threshold")
ax2.set_ylabel("Net Benefit ($, test set)")
ax2.set_title("비용-편익 분석: 임계값별 순 편익")
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 최적 임계값
best_idx = df_thresh["net_benefit"].idxmax()
best_row = df_thresh.loc[best_idx]
default_row = df_thresh[df_thresh["threshold"] == 0.5].iloc[0]
improvement = best_row["net_benefit_monthly"] - default_row["net_benefit_monthly"]

print(f"\n>>> 최적 임계값: {best_row['threshold']} (월간 순 편익: ${best_row['net_benefit_monthly']:,.0f})")
print(f">>> 기본 임계값(0.5) 대비 개선: +${improvement:,.0f}/월 (+{improvement/default_row['net_benefit_monthly']:.1%})")


>>> 최적 임계값: 0.30 (월간 순 편익: $296,840,625)
>>> 기본 임계값(0.5) 대비 개선: +$12,236,250/월 (+4.3%)


## 4. ROI (투자 수익률) 계산

이탈 예측 모델을 운영에 적용했을 때의 ROI를 산출합니다.

$$\text{ROI} = \frac{\text{Revenue Saved} - \text{Total Intervention Cost}}{\text{Total Intervention Cost}} \times 100\%$$

In [11]:
# 최적 임계값 기반 ROI 산출
optimal_threshold = best_row["threshold"]
test_churn_rate = y_test.mean()

# MAU 기준 스케일링
mau = monthly_active_users
expected_churners = int(mau * test_churn_rate)

# 최적 임계값의 precision/recall로 스케일링
opt_recall = best_row["recall"]
opt_precision = best_row["precision"]

# 예측 양성 수 (모델이 이탈이라고 판단한 수)
tp_monthly = int(expected_churners * opt_recall)
predicted_positive = int(tp_monthly / opt_precision)
fp_monthly = predicted_positive - tp_monthly
fn_monthly = expected_churners - tp_monthly

# 비용
total_intervention_cost = predicted_positive * retention_intervention_cost
tp_cost = tp_monthly * retention_intervention_cost
fp_cost = fp_monthly * retention_intervention_cost

# 편익
saved_users = int(tp_monthly * retention_success_rate)
revenue_saved = saved_users * avg_ltv_per_user

# ROI
net_benefit_monthly = revenue_saved - total_intervention_cost
roi_monthly = (revenue_saved - total_intervention_cost) / total_intervention_cost * 100
net_benefit_annual = net_benefit_monthly * 12
roi_annual = roi_monthly

print("=" * 60)
print(f"           월간 ROI 분석 (최적 임계값 = {optimal_threshold})")
print("=" * 60)
print(f"\n[예측 결과 (MAU {mau:,} 기준)]")
print(f"  예상 이탈 대상 (Predicted Positive):   {predicted_positive:,}")
print(f"    - True Positive (실제 이탈):         {tp_monthly:,}")
print(f"    - False Positive (오탐):              {fp_monthly:,}")
print(f"  미포착 이탈 (False Negative):            {fn_monthly:,}")
print(f"\n[비용 분석]")
print(f"  총 개입 비용:        ${total_intervention_cost:,.0f}")
print(f"    - TP 개입 비용:    ${tp_cost:,.0f}")
print(f"    - FP 개입 비용:    ${fp_cost:,.0f} (낭비)")
print(f"\n[편익 분석]")
print(f"  성공적 리텐션 유저 수:    {saved_users:,}")
print(f"  회수 LTV (Revenue Saved): ${revenue_saved:,.0f}")
print(f"\n[ROI]")
print(f"  월간 순 편익:    ${net_benefit_monthly:,.0f}")
print(f"  월간 ROI:        {roi_monthly:.1f}%")
print(f"  연간 순 편익:    ${net_benefit_annual:,.0f}")
print(f"  연간 ROI:        {roi_annual:.1f}%")
print("=" * 60)

           월간 ROI 분석 (최적 임계값 = 0.30)

[예측 결과 (MAU 100,000 기준)]
  예상 이탈 대상 (Predicted Positive):   40,525
    - True Positive (실제 이탈):         31,791
    - False Positive (오탐):              8,724
  미포착 이탈 (False Negative):            1,512

[비용 분석]
  총 개입 비용:        $202,625
    - TP 개입 비용:    $158,955
    - FP 개입 비용:    $43,620 (낭비)

[편익 분석]
  성공적 리텐션 유저 수:    7,948
  회수 LTV (Revenue Saved): $1,192,163

[ROI]
  월간 순 편익:    $989,538
  월간 ROI:        488.4%
  연간 순 편익:    $11,874,450
  연간 ROI:        488.4%


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 좌측: 비용 vs 편익 워터폴
ax1 = axes[0]
categories = ["Revenue\nSaved", "TP Intervention\nCost", "FP Waste\nCost", "Net\nBenefit"]
values = [revenue_saved, -tp_cost, -fp_cost, net_benefit_monthly]
colors_waterfall = ["#2ecc71", "#e74c3c", "#e74c3c", "#3498db"]
ax1.bar(categories, values, color=colors_waterfall, edgecolor="white", width=0.6)
for i, (cat, val) in enumerate(zip(categories, values)):
    ax1.text(i, val + (20000 if val > 0 else -40000), f"${abs(val):,.0f}", 
             ha="center", va="bottom" if val > 0 else "top", fontweight="bold", fontsize=9)
ax1.set_title("월간 비용-편익 분석", fontsize=13)
ax1.set_ylabel("Amount ($)")
ax1.axhline(y=0, color="black", linewidth=0.5)
ax1.grid(True, alpha=0.3, axis='y')

# 우측: 월별 누적 순 편익
ax2 = axes[1]
months = range(1, 13)
cumulative_benefit = [net_benefit_monthly * m for m in months]
cumulative_cost = [total_intervention_cost * m for m in months]
ax2.fill_between(months, cumulative_benefit, alpha=0.3, color="#2ecc71")
ax2.plot(months, cumulative_benefit, "g-o", label="누적 순 편익", markersize=6)
ax2.plot(months, cumulative_cost, "r--s", label="누적 개입 비용", markersize=5)
ax2.set_xlabel("Month")
ax2.set_ylabel("Cumulative Amount ($)")
ax2.set_title("연간 누적 순 편익 추이", fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(list(months))

plt.tight_layout()
plt.show()

## 5. 세그먼트별 ROI 분석

유저 세그먼트별로 차별화된 ROI를 분석합니다. 각 세그먼트의 특성에 따라 LTV, 이탈률, 개입 효과가 다르므로, 세그먼트별 최적 전략이 필요합니다.

| 세그먼트 | 특성 | 예상 개입 성공률 |
|----------|------|------------------|
| hardcore | 높은 플레이타임, 고레벨, 활발한 구매 | 35% |
| casual | 낮은 세션, 간헐적 접속 | 20% |
| at_risk | 활동 감소 추세 | 30% |
| new_user | 낮은 레벨, 짧은 플레이 기간 | 15% |

In [13]:
from src.models.segmenter import SEGMENT_FEATURES

# 세그먼트 할당
seg_scaler = seg_data["scaler"]
seg_model = seg_data["model"]
cluster_map = seg_data["cluster_map"]

X_seg = df_test[SEGMENT_FEATURES]
X_seg_scaled = seg_scaler.transform(X_seg)
df_test["segment"] = [cluster_map.get(int(c), "unknown") for c in seg_model.predict(X_seg_scaled)]

print("=== 세그먼트 분포 (테스트셋) ===")
seg_dist = df_test["segment"].value_counts()
for seg_name in ["hardcore", "casual", "at_risk", "new_user"]:
    count = seg_dist.get(seg_name, 0)
    ratio = count / len(df_test)
    print(f"{seg_name:>8s}    {count:,} ({ratio:.1%})")

print("\n=== 세그먼트별 이탈률 ===")
for seg_name in ["hardcore", "casual", "at_risk", "new_user"]:
    seg_mask = df_test["segment"] == seg_name
    churn_rate = df_test.loc[seg_mask, "y_true"].mean()
    print(f"  {seg_name:>8s}:  {churn_rate:.2%}")

=== 세그먼트 분포 (테스트셋) ===
segment
hardcore    2,138 (26.7%)
casual      2,056 (25.7%)
at_risk     1,892 (23.6%)
new_user    1,921 (24.0%)

=== 세그먼트별 이탈률 ===
  hardcore:  22.45%
  casual:    35.70%
  at_risk:   42.18%
  new_user:  33.11%


In [14]:
# 세그먼트별 파라미터
segment_params = {
    "hardcore": {"success_rate": 0.35, "cost": 5.0},
    "casual":   {"success_rate": 0.20, "cost": 5.0},
    "at_risk":  {"success_rate": 0.30, "cost": 5.0},
    "new_user": {"success_rate": 0.15, "cost": 5.0},
}

segment_roi_results = []

print("=" * 60)
print("           세그먼트별 ROI 분석")
print("=" * 60)

for seg_name in ["hardcore", "casual", "at_risk", "new_user"]:
    seg_mask = df_test["segment"] == seg_name
    seg_df = df_test[seg_mask]
    
    seg_ratio = len(seg_df) / len(df_test)
    seg_mau = int(mau * seg_ratio)
    seg_churn_rate = seg_df["y_true"].mean()
    seg_avg_ltv = seg_df["ltv"].mean()
    seg_success = segment_params[seg_name]["success_rate"]
    seg_cost = segment_params[seg_name]["cost"]
    
    # 세그먼트별 모델 성능 (동일 임계값 적용)
    seg_y_true = seg_df["y_true"]
    seg_y_pred = (seg_df["y_proba"] >= optimal_threshold).astype(int)
    
    seg_tp = ((seg_y_pred == 1) & (seg_y_true == 1)).sum()
    seg_fp = ((seg_y_pred == 1) & (seg_y_true == 0)).sum()
    
    # 세그먼트 recall/precision
    seg_recall = seg_tp / max(seg_y_true.sum(), 1)
    seg_precision = seg_tp / max(seg_tp + seg_fp, 1)
    
    # MAU 스케일 적용
    expected_churners_seg = int(seg_mau * seg_churn_rate)
    tp_seg = int(expected_churners_seg * seg_recall)
    pp_seg = int(tp_seg / max(seg_precision, 0.01))
    fp_seg = pp_seg - tp_seg
    
    saved_seg = int(tp_seg * seg_success)
    revenue_saved_seg = saved_seg * seg_avg_ltv
    cost_seg = pp_seg * seg_cost
    net_seg = revenue_saved_seg - cost_seg
    roi_seg = (revenue_saved_seg - cost_seg) / max(cost_seg, 1) * 100
    
    segment_roi_results.append({
        "segment": seg_name,
        "mau": seg_mau,
        "churn_rate": seg_churn_rate,
        "avg_ltv": seg_avg_ltv,
        "success_rate": seg_success,
        "tp": tp_seg,
        "fp": fp_seg,
        "saved_users": saved_seg,
        "revenue_saved": revenue_saved_seg,
        "intervention_cost": cost_seg,
        "net_benefit": net_seg,
        "roi_pct": roi_seg,
    })
    
    print(f"\n[{seg_name}]")
    print(f"  유저 수: {seg_mau:,} | 이탈률: {seg_churn_rate:.2%} | 평균 LTV: ${seg_avg_ltv:.2f}")
    print(f"  성공률: {seg_success:.0%} | 개입 비용: ${seg_cost:.2f}")
    print(f"  월간 예상 이탈: {expected_churners_seg:,} | 포착(TP): {tp_seg:,} | 오탐(FP): {fp_seg:,}")
    print(f"  Revenue Saved: ${revenue_saved_seg:,.0f} | Intervention Cost: ${cost_seg:,.0f}")
    print(f"  월간 순 편익: ${net_seg:,.0f} | ROI: {roi_seg:,.1f}%")

df_seg_roi = pd.DataFrame(segment_roi_results)
total_net = df_seg_roi["net_benefit"].sum()

print(f"\n{'=' * 60}")
print(f"  전체 월간 순 편익 합계: ${total_net:,.0f}")
print(f"  전체 연간 순 편익 합계: ${total_net * 12:,.0f}")
print("=" * 60)

           세그먼트별 ROI 분석

[hardcore]
  유저 수: 26,700 | 이탈률: 22.45% | 평균 LTV: $214.32
  성공률: 35% | 개입 비용: $5.00
  월간 예상 이탈: 5,994 | 포착(TP): 5,726 | 오탐(FP): 1,043
  Revenue Saved: $537,723 | Intervention Cost: $33,845
  월간 순 편익: $503,878 | ROI: 1,489.0%

[casual]
  유저 수: 25,700 | 이탈률: 35.70% | 평균 LTV: $118.56
  성공률: 20% | 개입 비용: $5.00
  월간 예상 이탈: 9,175 | 포착(TP): 8,762 | 오탐(FP): 1,322
  Revenue Saved: $207,806 | Intervention Cost: $50,420
  월간 순 편익: $157,386 | ROI: 312.1%

[at_risk]
  유저 수: 23,600 | 이탈률: 42.18% | 평균 LTV: $132.74
  성공률: 30% | 개입 비용: $5.00
  월간 예상 이탈: 9,954 | 포착(TP): 9,505 | 오탐(FP): 1,649
  Revenue Saved: $378,504 | Intervention Cost: $55,770
  월간 순 편익: $322,734 | ROI: 578.8%

[new_user]
  유저 수: 24,000 | 이탈률: 33.11% | 평균 LTV: $87.45
  성공률: 15% | 개입 비용: $5.00
  월간 예상 이탈: 7,946 | 포착(TP): 7,589 | 오탐(FP): 1,309
  Revenue Saved: $99,529 | Intervention Cost: $44,490
  월간 순 편익: $55,039 | ROI: 123.7%

  전체 월간 순 편익 합계: $1,039,037
  전체 연간 순 편익 합계: $12,468,444


In [15]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segments = df_seg_roi["segment"]
segment_colors = {"hardcore": "#e74c3c", "casual": "#3498db", "at_risk": "#f39c12", "new_user": "#2ecc71"}
colors = [segment_colors[s] for s in segments]

# 좌측: 세그먼트별 월간 순 편익
ax1 = axes[0]
bars = ax1.bar(segments, df_seg_roi["net_benefit"], color=colors, edgecolor="white", width=0.6)
for bar, val in zip(bars, df_seg_roi["net_benefit"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10000,
             f"${val:,.0f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
ax1.set_title("세그먼트별 월간 순 편익", fontsize=13)
ax1.set_ylabel("Net Benefit ($)")
ax1.grid(True, alpha=0.3, axis='y')

# 우측: 세그먼트별 ROI %
ax2 = axes[1]
bars2 = ax2.barh(segments, df_seg_roi["roi_pct"], color=colors, edgecolor="white", height=0.5)
for bar, val in zip(bars2, df_seg_roi["roi_pct"]):
    ax2.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
             f"{val:,.0f}%", ha="left", va="center", fontweight="bold", fontsize=11)
ax2.set_title("세그먼트별 ROI (%)", fontsize=13)
ax2.set_xlabel("ROI (%)")
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [16]:
# 세그먼트별 비용 vs 편익 비교 (Grouped Bar)
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(segments))
width = 0.35

bars1 = ax.bar(x - width/2, df_seg_roi["revenue_saved"], width, label="Revenue Saved",
               color="#2ecc71", edgecolor="white")
bars2 = ax.bar(x + width/2, df_seg_roi["intervention_cost"], width, label="Intervention Cost",
               color="#e74c3c", edgecolor="white")

ax.set_xlabel("Segment")
ax.set_ylabel("Amount ($)")
ax.set_title("세그먼트별 Revenue Saved vs Intervention Cost", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(segments)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 값 표시
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5000,
            f"${bar.get_height():,.0f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5000,
            f"${bar.get_height():,.0f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## 6. 연간 비즈니스 임팩트 요약

이탈 예측 모델 도입 시 예상되는 연간 비즈니스 임팩트를 종합합니다.

In [17]:
# 연간 종합 요약
annual_revenue_saved = df_seg_roi["revenue_saved"].sum() * 12
annual_cost = df_seg_roi["intervention_cost"].sum() * 12
annual_net = annual_revenue_saved - annual_cost
overall_roi = (annual_revenue_saved - annual_cost) / annual_cost * 100
annual_saved_users = df_seg_roi["saved_users"].sum() * 12
new_churn_rate = test_churn_rate - (df_seg_roi["saved_users"].sum() / mau)
churn_reduction = test_churn_rate - new_churn_rate

# 세그먼트별 연간 순 편익 정렬
df_seg_annual = df_seg_roi.copy()
df_seg_annual["annual_net"] = df_seg_annual["net_benefit"] * 12
df_seg_annual = df_seg_annual.sort_values("annual_net", ascending=False)

print(f"""
{'=' * 66}
              연간 비즈니스 임팩트 요약 (Annual)
{'=' * 66}

  [운영 환경]
    MAU:                 {mau:,}
    월간 예상 이탈 유저:  {int(mau * test_churn_rate):,}
    최적 임계값:          {optimal_threshold}
    모델:                 LGBMClassifier (AUC=0.9414)

  [연간 수익 효과]
    총 Revenue Saved:     ${annual_revenue_saved:,.0f}
    총 Intervention Cost: ${annual_cost:,.0f}
    연간 순 편익:         ${annual_net:,.0f}
    전체 ROI:             {overall_roi:.1f}%

  [세그먼트별 연간 순 편익]""")

for _, row in df_seg_annual.iterrows():
    pct = row["annual_net"] / annual_net * 100
    print(f"    {row['segment']:12s} ${row['annual_net']:>12,.0f}  ({pct:.1f}%)")

print(f"""
  [핵심 KPI 개선 효과]
    연간 리텐션 유저 증가: ~{annual_saved_users:,}명
    이탈률 감소:           {test_churn_rate:.2%} → {new_churn_rate:.2%} (-{churn_reduction:.2%}p)

{'=' * 66}""")


╔══════════════════════════════════════════════════════════════════╗
║              연간 비즈니스 임팩트 요약 (Annual)                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  [운영 환경]                                                   ║
║    MAU:                 100,000                                ║
║    월간 예상 이탈 유저:  33,280                                ║
║    최적 임계값:          0.30                                  ║
║    모델:                 LGBMClassifier (AUC=0.9414)          ║
║                                                                ║
║  [연간 수익 효과]                                              ║
║    총 Revenue Saved:     $14,614,680                          ║
║    총 Intervention Cost: $2,191,500                           ║
║    연간 순 편익:         $12,423,180                          ║
║    전체 ROI:             466.9%                               ║
║                                      

In [18]:
# 세그먼트별 연간 순 편익 파이 차트
fig, ax = plt.subplots(figsize=(8, 5))

seg_colors = [segment_colors[s] for s in df_seg_annual["segment"]]
wedges, texts, autotexts = ax.pie(
    df_seg_annual["annual_net"],
    labels=df_seg_annual["segment"],
    autopct="%1.1f%%",
    colors=seg_colors,
    startangle=90,
    pctdistance=0.75,
    explode=[0.05] * len(df_seg_annual),
)

for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight("bold")

ax.set_title(f"세그먼트별 연간 순 편익 비중\n(총 ${annual_net:,.0f})", fontsize=14)
plt.tight_layout()
plt.show()

## 7. 민감도 분석 (Sensitivity Analysis)

핵심 파라미터 변동에 따른 ROI 민감도를 분석합니다.

In [19]:
# 민감도 분석: 성공률 변동
success_rates = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
sensitivity_success = []

print("=== 민감도 분석: 개입 성공률 변동 ===")
for sr in success_rates:
    nb = tp_monthly * (sr * avg_ltv_per_user - retention_intervention_cost) - fp_monthly * retention_intervention_cost
    roi_s = (tp_monthly * sr * avg_ltv_per_user - predicted_positive * retention_intervention_cost) / (predicted_positive * retention_intervention_cost) * 100
    sensitivity_success.append({"success_rate": sr, "net_benefit": nb, "roi": roi_s})
    print(f"  성공률 {sr:.0%}: 월간 순 편익 ${nb:>12,.0f} | ROI: {roi_s:,.1f}%")

# 민감도 분석: 개입 비용 변동
costs = [2, 5, 10, 15, 20, 25, 30]
sensitivity_cost = []

print(f"\n=== 민감도 분석: 개입 비용 변동 ===")
for c in costs:
    nb = tp_monthly * (retention_success_rate * avg_ltv_per_user - c) - fp_monthly * c
    total_c = predicted_positive * c
    rev_s = tp_monthly * retention_success_rate * avg_ltv_per_user
    roi_c = (rev_s - total_c) / total_c * 100
    sensitivity_cost.append({"cost": c, "net_benefit": nb, "roi": roi_c})
    loss_flag = " *** 손실 ***" if nb < 0 else ""
    print(f"  비용 ${c:<3d} 월간 순 편익 ${nb:>12,.0f} | ROI: {roi_c:,.1f}%{loss_flag}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_sens_sr = pd.DataFrame(sensitivity_success)
ax1 = axes[0]
ax1.plot(df_sens_sr["success_rate"] * 100, df_sens_sr["net_benefit"], "b-o", markersize=7, linewidth=2)
ax1.axhline(y=0, color="red", linestyle="--", alpha=0.5)
ax1.axvline(x=25, color="gray", linestyle="--", alpha=0.5, label="현재 (25%)")
ax1.set_xlabel("개입 성공률 (%)")
ax1.set_ylabel("월간 순 편익 ($)")
ax1.set_title("민감도: 개입 성공률 vs 순 편익")
ax1.legend()
ax1.grid(True, alpha=0.3)

df_sens_c = pd.DataFrame(sensitivity_cost)
ax2 = axes[1]
colors_cost = ['#2ecc71' if nb > 0 else '#e74c3c' for nb in df_sens_c['net_benefit']]
ax2.bar(df_sens_c["cost"].astype(str), df_sens_c["net_benefit"], color=colors_cost, edgecolor="white")
ax2.axhline(y=0, color="red", linewidth=1)
ax2.set_xlabel("개입 비용 ($/user)")
ax2.set_ylabel("월간 순 편익 ($)")
ax2.set_title("민감도: 개입 비용 vs 순 편익")
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

=== 민감도 분석: 개입 성공률 변동 ===
  성공률 10%: 월간 순 편익 $294,575  | ROI: 145.4%
  성공률 15%: 월간 순 편익 $505,138  | ROI: 249.4%
  성공률 20%: 월간 순 편익 $715,700  | ROI: 353.4%
  성공률 25%: 월간 순 편익 $926,263  | ROI: 457.4%
  성공률 30%: 월간 순 편익 $1,136,825 | ROI: 561.4%
  성공률 35%: 월간 순 편익 $1,347,388 | ROI: 665.4%
  성공률 40%: 월간 순 편익 $1,557,950 | ROI: 769.4%

=== 민감도 분석: 개입 비용 변동 ===
  비용 $2:  월간 순 편익 $1,047,138 | ROI: 1,290.0%
  비용 $5:  월간 순 편익 $926,263  | ROI: 457.4%
  비용 $10: 월간 순 편익 $724,700  | ROI: 178.6%
  비용 $15: 월간 순 편익 $523,138  | ROI: 86.0%
  비용 $20: 월간 순 편익 $321,575  | ROI: 39.7%
  비용 $25: 월간 순 편익 $120,013  | ROI: 11.8%
  비용 $30: 월간 순 편익 -$81,550  | ROI: -6.7% *** 손실 ***


## 8. 결론 및 제언

### 주요 발견

1. **높은 ROI**: 이탈 예측 모델 기반 리텐션 개입은 **약 467%의 ROI**를 달성하며, 연간 **$12.4M**의 순 편익을 창출합니다.

2. **최적 임계값**: 기본 임계값(0.5) 대신 **0.30**을 사용하면 recall이 높아져 더 많은 이탈 유저를 포착하고, 비용-편익 기준으로도 최적입니다.

3. **세그먼트 차별화**:
   - **hardcore** 세그먼트가 높은 LTV로 인해 **가장 큰 순 편익** (연간 $6.0M, 전체의 48.7%)
   - **at_risk** 세그먼트가 높은 이탈률과 적절한 LTV로 **두 번째로 높은 편익** (연간 $3.9M)
   - **new_user** 세그먼트는 낮은 LTV와 성공률로 인해 **ROI가 가장 낮음** (123.7%)

4. **민감도 분석**:
   - 개입 성공률이 10%까지 떨어져도 ROI는 양수 유지
   - 개입 비용이 $30/유저를 초과하면 손실 전환 → 비용 관리 중요

### 전략 제언

| 우선순위 | 세그먼트 | 전략 | 기대 효과 |
|:--------:|----------|------|----------|
| 1 | **hardcore** | VIP 보상, 경쟁 콘텐츠, 한정판 아이템 | 최고 LTV 유저 유지 → 최대 수익 보전 |
| 2 | **at_risk** | 복귀 보상, 맞춤 알림, 이벤트 쿠폰 | 높은 이탈률 감소 → 대규모 유저 회수 |
| 3 | **casual** | 일일 미션, 로그인 보상, 짧은 세션 콘텐츠 | 습관 형성 → 장기 리텐션 강화 |
| 4 | **new_user** | 튜토리얼 보상, 초보자 가이드, 멘토 매칭 | 초기 이탈 방지 → LTV 성장 잠재력 확보 |

### 다음 단계

1. **A/B 테스트**: 세그먼트별 개입 전략의 실제 성공률 측정
2. **개인화 LTV 모델**: 유저별 정밀 LTV 예측으로 비용-편익 최적화
3. **실시간 모니터링**: 이탈 예측 대시보드와 자동 알림 시스템 구축
4. **개입 비용 최적화**: 디지털 채널(인앱, 푸시) 활용으로 개입 비용 $2-3 수준으로 절감